# Lab: prevendo a sobrevivência no Titanic

Notebook de aula da página *Regressão Logística*: https://insper.github.io/ML/pt/2026.2/classes/logistic-regression/

A lista de passageiros do Titanic da aula de [pré-processamento](../preprocessing/index.md), agora com uma pergunta: *quem sobreviveu?* O lab aplica tudo o que foi visto a dados reais e bagunçados: um pipeline com imputação, codificação e escalonamento; probabilidades conferidas à mão; coeficientes lidos como razões de chances; e uma estimativa honesta da acurácia em passageiros que o modelo nunca viu. O modelo é a regressão logística pura desta aula (`C=np.inf`), então não há nada para ajustar: dividimos os dados uma vez em **treino** (ajustar e estudar o modelo) e **teste** (reportar, uma vez, no fim).

Rode as células em ordem. Em cada passo, responda o **Preveja** com a dupla *antes* de rodar. Nas **Simulações**, escreva o experimento na célula vazia; as soluções e a leitura de cada resultado estão na página da aula.

## Passo 1 — carregar e olhar

**Preveja.** A taxa de sobrevivência geral fica acima ou abaixo de 50%? Quem sobreviveu mais: as mulheres da 3ª classe ou os homens da 1ª?

**Resposta (antes de rodar):** A taxa geral deve ficar bem abaixo de 50% — o Titanic é
lembrado justamente por mais gente ter morrido do que sobrevivido, então chuto algo perto de
35-40%. Quanto a quem sobrevive mais: a política "mulheres e crianças primeiro" deveria valer
mais que a classe social, então espero que as mulheres da 3ª classe sobrevivam mais que os homens
da 1ª classe, mesmo a 3ª classe sendo mais pobre.


In [1]:
import numpy as np
import pandas as pd

url = "https://raw.githubusercontent.com/hsandmann/biblio/refs/heads/main/ml/aula04/titanic/train.csv"
df = pd.read_csv(url)
print(df.shape)
print(f"taxa de sobrevivência: {df['Survived'].mean():.3f}")
print(df.pivot_table(index='Sex', columns='Pclass', values='Survived', aggfunc='mean').round(3))
print("faltantes:", df[['Age', 'Embarked', 'Cabin']].isna().sum().to_dict())

(891, 12)
taxa de sobrevivência: 0.384
Pclass      1      2      3
Sex                        
female  0.968  0.921  0.500
male    0.369  0.157  0.135
faltantes: {'Age': 177, 'Embarked': 2, 'Cabin': 687}


**Resposta (depois de rodar):** Confirmado — a taxa geral é **38,4%** (bem abaixo de 50%,
como previsto). Na tabela sexo × classe, as mulheres da 3ª classe sobrevivem em **50,0%** dos
casos, contra apenas **36,9%** dos homens da 1ª classe — a hipótese se confirma: sexo pesa mais
que classe (mulheres da pior classe ainda sobrevivem mais que homens da melhor classe). Faltam
177 valores de `Age`, 2 de `Embarked` e 687 (77,1%) de `Cabin`.


## Passo 2 — treino e teste

**Preveja.** Por que separar o conjunto de teste agora, antes de montar qualquer modelo? Por que não deixar isso para o fim?

**Resposta (antes de rodar):** Porque qualquer decisão tomada olhando para todos os dados —
imputar a idade com a mediana, escolher quais colunas usar, até só "espiar" os números — já é uma
forma de deixar informação do conjunto de teste influenciar o modelo (vazamento/*data leakage*).
Separar o teste antes garante que ele simule o futuro: dados que o modelo nunca viu, nem de forma
indireta pelas estatísticas usadas no pré-processamento.


In [2]:
from sklearn.model_selection import train_test_split

features = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked']
X, y = df[features], df['Survived']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print(len(X_train), len(X_test))
print(f"taxa de sobrevivência: treino {y_train.mean():.3f} · teste {y_test.mean():.3f}")

712 179
taxa de sobrevivência: treino 0.383 · teste 0.385


**Resposta (depois de rodar):** O split ficou com **712** linhas de treino e **179** de
teste (80/20 de 891), com `stratify=y` preservando a taxa de sobrevivência quase igual nos dois
lados: **38,3%** no treino e **38,5%** no teste — a diferença é de só 0,2 pontos percentuais,
exatamente o que se espera de um split estratificado.


## Passo 3 — uma baseline

**Preveja.** Um "modelo" que responde sempre "não sobreviveu", sem olhar o passageiro, acerta mais ou menos da metade das vezes? Por quê?

**Resposta (antes de rodar):** Bem mais da metade, não perto de 50%. Como a maioria dos
passageiros não sobreviveu (classe desbalanceada), um "modelo" que chuta sempre a classe
majoritária acerta a própria proporção da maioria — algo perto de 60-65%, não 50%, porque a base
não é balanceada.


In [3]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier(strategy='most_frequent').fit(X_train, y_train)
print(f"sempre 'não sobreviveu': acurácia de treino {dummy.score(X_train, y_train):.3f}")

sempre 'não sobreviveu': acurácia de treino 0.617


**Resposta (depois de rodar):** Confirmado — a acurácia de treino do `DummyClassifier` é
**61,7%**, bem acima de 50% e quase idêntica a 1 − 0,383 (a fração de não sobreviventes no
treino). É só a classe majoritária "sem olhar passageiro algum" — qualquer modelo real precisa
superar esses 61,7% para valer a pena.


## Passo 4 — o pipeline

**Preveja.** `Pclass` vale 1, 2 ou 3. Por que tratá-la como categoria (uma coluna one-hot para cada classe) e não como um número?

**Resposta (antes de rodar):** Porque o efeito de `Pclass` na sobrevivência não deveria ser
linear no número da classe: a diferença entre 1ª e 2ª classe não é necessariamente igual à
diferença entre 2ª e 3ª. Tratar como número força o modelo a usar um único coeficiente
proporcional a 1, 2, 3; como categoria (one-hot), cada classe recebe seu próprio efeito livre.


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric = ['Age', 'SibSp', 'Parch', 'Fare']
categorical = ['Pclass', 'Sex', 'Embarked']

preprocess = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'),
                          OneHotEncoder(drop='first')), categorical)])
model = make_pipeline(preprocess, LogisticRegression(C=np.inf, max_iter=1000))   # C=np.inf: sem penalidade
model.fit(X_train, y_train)

enc = model[0].named_transformers_['cat'][-1]
print("categorias de referência (descartadas):", {c: cats[0] for c, cats in zip(categorical, enc.categories_)})
print(f"{model[-1].coef_.size} coeficientes · acurácia de treino {model.score(X_train, y_train):.3f}")

categorias de referência (descartadas): {'Pclass': 1, 'Sex': 'female', 'Embarked': 'C'}
9 coeficientes · acurácia de treino 0.809


**Resposta (depois de rodar):** Confirmado — o pipeline usa `Pclass`, `Sex` e `Embarked` como
categóricas (categoria de referência descartada: Pclass 1, Sex female, Embarked C) e chega a
**9 coeficientes** e acurácia de treino **80,9%**, bem acima dos 61,7% da baseline. Olhando a
tabela do Passo 1, a queda de sobrevivência feminina entre 1ª (96,8%) e 2ª classe (92,1%) é de só
4,7 p.p., enquanto entre 2ª e 3ª é de 42,1 p.p. — nada parecido com um efeito linear em 1, 2, 3,
confirmando por que tratar como categoria fazia sentido.


## Passo 5 — probabilidades para dois passageiros

**Preveja.** Olhando a tabela do passo 1: um homem da 3ª classe e uma mulher da 1ª classe. Quem recebe probabilidade alta e quem recebe baixa?

**Resposta (antes de rodar):** Pela tabela do Passo 1, o homem da 3ª classe deve receber
probabilidade baixa (perto de 13,5%, a taxa de sobrevivência de homens da 3ª classe) e a mulher da
1ª classe deve receber probabilidade alta (perto de 96,8%, a taxa de mulheres da 1ª classe).


In [5]:
passengers = pd.DataFrame({'Pclass': [3, 1], 'Sex': ['male', 'female'], 'Age': [25, 30],
                           'SibSp': [0, 1], 'Parch': [0, 0], 'Fare': [8.05, 80.0],
                           'Embarked': ['S', 'C']})
print(model.predict_proba(passengers).round(3))
print(model.predict(passengers))

[[0.907 0.093]
 [0.058 0.942]]
[0 1]


**Resposta (depois de rodar):** Confirmado — o homem da 3ª classe recebe probabilidade de
sobrevivência **9,3%** (`predict` = 0, não sobreviveu) e a mulher da 1ª classe recebe **94,2%**
(`predict` = 1, sobreviveu). Os números não são idênticos aos da tabela bruta (13,5%/96,8%) porque
o modelo também usa idade, tarifa e demais colunas desses dois passageiros específicos, mas ficam
bem próximos e na mesma direção.


**Simulação: o efeito da idade.** Tome um passageiro da 2ª classe que viaja com os pais (`Parch = 2`) e um irmão (`SibSp = 1`), passagem de 30 libras, embarcado em Southampton. Mude só a idade (2, 8, 15, 25, 40 e 60 anos), primeiro para um menino e depois para uma menina. Observe a probabilidade prevista. Depois confira nas linhas de treino quantas crianças de até 12 anos da 2ª classe sobreviveram.

In [6]:
ages = [2, 8, 15, 25, 40, 60]
rows = []
for sex in ['male', 'female']:
    for age in ages:
        rows.append({'Pclass': 2, 'Sex': sex, 'Age': age, 'SibSp': 1, 'Parch': 2,
                     'Fare': 30.0, 'Embarked': 'S'})
sim_df = pd.DataFrame(rows)
sim_df['proba_sobreviver'] = model.predict_proba(sim_df)[:, 1].round(3)
print(sim_df)

train_full = X_train.copy()
train_full['Survived'] = y_train
kids = train_full[(train_full['Pclass'] == 2) & (train_full['Age'] <= 12)]
print(f"crianças <=12 anos, 2ª classe, no treino: {len(kids)} · sobreviventes: {kids['Survived'].sum()} · taxa: {kids['Survived'].mean():.3f}")


    Pclass     Sex  Age  SibSp  Parch  Fare Embarked  proba_sobreviver
0        2    male    2      1      2  30.0        S             0.380
1        2    male    8      1      2  30.0        S             0.326
2        2    male   15      1      2  30.0        S             0.268
3        2    male   25      1      2  30.0        S             0.198
4        2    male   40      1      2  30.0        S             0.120
5        2    male   60      1      2  30.0        S             0.059
6        2  female    2      1      2  30.0        S             0.900
7        2  female    8      1      2  30.0        S             0.876
8        2  female   15      1      2  30.0        S             0.843
9        2  female   25      1      2  30.0        S             0.784
10       2  female   40      1      2  30.0        S             0.667
11       2  female   60      1      2  30.0        S             0.477
crianças <=12 anos, 2ª classe, no treino: 15 · sobreviventes: 15 · taxa: 1.00

**Resposta:** Para o menino de 2ª classe, a probabilidade cai suavemente com a idade: **38,0%**
(2 anos), 32,6% (8), 26,8% (15), 19,8% (25), 12,0% (40) até **5,9%** (60 anos) — uma queda de 32,1
pontos percentuais do mais novo ao mais velho. Para a menina da mesma família, o padrão é o mesmo
formato, só deslocado bem mais para cima: **90,0%** (2 anos) até **47,7%** (60 anos), uma queda de
42,3 pontos. Em nenhuma idade o menino supera a menina — o efeito do sexo (coeficiente muito mais
negativo para `male`) domina o efeito da idade nessa faixa. Nas linhas de treino, todas as
**15** crianças de até 12 anos da 2ª classe sobreviveram (taxa de 100%) — consistente com a
probabilidade alta (>87%) que o modelo atribui às crianças pequenas dessa classe.


## Passo 6 — a probabilidade do homem, à mão

**Preveja.** Olhando os coeficientes que vão aparecer, qual atributo você espera que mais derrube o escore do homem?

**Resposta (antes de rodar):** Pela magnitude dos efeitos vistos até aqui (sexo parece
dominar mais que classe ou idade na tabela do Passo 1), chuto que o atributo `Sex_male` (ser
homem) vai ter o maior peso negativo e será o que mais derruba o escore do homem da 3ª classe.


In [7]:
names = model[0].get_feature_names_out()
w, b = model[-1].coef_[0], model[-1].intercept_[0]
x_man = model[0].transform(passengers)[0]            # o homem da 3ª classe, depois do pré-processamento
print(pd.DataFrame({'x': x_man, 'w': w, 'w*x': w * x_man}, index=names).round(3))
z = b + (w * x_man).sum()
print(f"z = {b:.3f} + {(w * x_man).sum():.3f} = {z:.3f}   sigmoid(z) = {1 / (1 + np.exp(-z)):.3f}")

                     x      w    w*x
num__Age        -0.350 -0.513  0.180
num__SibSp      -0.465 -0.267  0.124
num__Parch      -0.466 -0.072  0.034
num__Fare       -0.495  0.108 -0.053
cat__Pclass_2    0.000 -0.987 -0.000
cat__Pclass_3    1.000 -2.239 -2.239
cat__Sex_male    1.000 -2.685 -2.685
cat__Embarked_Q  0.000  0.320  0.000
cat__Embarked_S  1.000 -0.417 -0.417
z = 2.782 + -5.057 = -2.275   sigmoid(z) = 0.093


**Resposta (depois de rodar):** Confirmado — na tabela de contribuições, `cat__Sex_male`
contribui com **w·x = -2,685** (o dummy vale 1 para esse passageiro), a maior queda em módulo de
toda a tabela, maior até que `cat__Pclass_3` (**-2,239**). Somando tudo, z = 2,782 + (-5,057) =
**-2,275**, e sigmoid(-2,275) = **0,093** — exatamente os 9,3% de probabilidade de sobrevivência
já vistos no Passo 5 para esse mesmo passageiro, batendo as duas contas.


**Simulação: um atributo por vez.** Parta do homem da 3ª classe e transforme-o, um passo de cada vez, na passageira de referência: primeiro mude o sexo para `female`, depois a classe para 1, depois o porto para `C`. A cada passo, imprima o escore \(z\) (`model.decision_function`) e a probabilidade. Quanto \(z\) muda em cada passo? E a probabilidade?

In [8]:
man3 = passengers.iloc[[0]].copy()
steps = []
cur = man3.copy()
steps.append(('homem 3a classe (base)', cur.copy()))
cur = cur.copy(); cur['Sex'] = 'female'
steps.append(('sexo -> female', cur.copy()))
cur = cur.copy(); cur['Pclass'] = 1
steps.append(('classe -> 1', cur.copy()))
cur = cur.copy(); cur['Embarked'] = 'C'
steps.append(('embarked -> C', cur.copy()))

prev_z = None
for label, row in steps:
    z_now = model.decision_function(row)[0]
    p_now = model.predict_proba(row)[0, 1]
    delta = None if prev_z is None else z_now - prev_z
    delta_str = f"{delta:+.3f}" if delta is not None else "—"
    print(f"{label:28s} z={z_now:7.3f} (delta z {delta_str})  p={p_now:.3f}")
    prev_z = z_now


homem 3a classe (base)       z= -2.275 (delta z —)  p=0.093
sexo -> female               z=  0.410 (delta z +2.685)  p=0.601
classe -> 1                  z=  2.649 (delta z +2.239)  p=0.934
embarked -> C                z=  3.066 (delta z +0.417)  p=0.955


**Resposta:** Partindo do homem da 3ª classe (z = -2,275, p = 0,093): trocar o sexo para
`female` soma **+2,685** ao escore (z = 0,410, p = 0,601) — é o próprio coeficiente de
`Sex_male`, com sinal trocado, porque sair de x=1 para x=0 remove exatamente w·1. Mudar a classe
de 3 para 1 soma **+2,239** (z = 2,649, p = 0,934) — de novo, o coeficiente de `Pclass_3` some
porque a referência (`Pclass_1`) não tem dummy próprio. Por fim, mudar o porto para `C` soma
**+0,417** (z = 3,066, p = 0,955) — o menor dos três saltos. A ordem dos saltos em z (2,685 >
2,239 > 0,417) é exatamente a ordem das magnitudes dos coeficientes envolvidos, confirmando que
cada passo muda z pela diferença exata entre os dois coeficientes de dummy trocados (ou, para a
referência, pelo próprio coeficiente). Em probabilidade, o primeiro passo (sexo) é o que mais
importa (+0,508 em p), porque a sigmoide é mais inclinada perto de z=0; o terceiro passo (porto),
já perto da saturação (p>0,93), move p muito pouco (+0,021) mesmo com z subindo.


## Passo 7 — coeficientes como razões de chances

**Preveja.** O coeficiente de `Age` é negativo. Então, com o resto fixo, um passageiro mais velho tem chance maior ou menor de sobreviver? A razão de chances $e^{\text{coef}}$ dele fica acima ou abaixo de 1?

**Resposta (antes de rodar):** Coeficiente negativo em `Age` significa que, com tudo mais
fixo, aumentar a idade reduz o escore z e portanto reduz a probabilidade de sobreviver (passageiros
mais velhos sobrevivem menos). A razão de chances $e^{\text{coef}}$ de um coeficiente negativo é
sempre menor que 1 (já que $e^x < 1$ para $x<0$), então deve ficar abaixo de 1.


In [9]:
coef = pd.Series(w, index=names)
print(pd.DataFrame({'coef': coef, 'e^coef': np.exp(coef)}).round(3))
sd = model[0].named_transformers_['num'][-1].scale_
print("um desvio-padrão:", ", ".join(f"{c} {s:.1f}" for c, s in zip(numeric, sd)))

                  coef  e^coef
num__Age        -0.513   0.599
num__SibSp      -0.267   0.765
num__Parch      -0.072   0.931
num__Fare        0.108   1.114
cat__Pclass_2   -0.987   0.373
cat__Pclass_3   -2.239   0.107
cat__Sex_male   -2.685   0.068
cat__Embarked_Q  0.320   1.377
cat__Embarked_S -0.417   0.659
um desvio-padrão: Age 13.0, SibSp 1.1, Parch 0.8, Fare 48.0


In [10]:
print(f"mulher vs homem:  chance x {np.exp(-coef['cat__Sex_male']):.2f}")
print(f"1ª vs 3ª classe:  chance x {np.exp(-coef['cat__Pclass_3']):.2f}")
print(f"2ª vs 3ª classe:  chance x {np.exp(coef['cat__Pclass_2'] - coef['cat__Pclass_3']):.2f}")

mulher vs homem:  chance x 14.66
1ª vs 3ª classe:  chance x 9.39
2ª vs 3ª classe:  chance x 3.50


**Resposta (depois de rodar):** Confirmado — `Age` tem coeficiente **-0,513** e razão de
chances $e^{-0,513}=$ **0,599**: por desvio-padrão de idade (13,0 anos), a chance de sobreviver
multiplica por 0,599 (cai ~40%). Esse valor bate exatamente com a página (0,599). As outras razões
de chances também batem com a página: mulher vs. homem **14,66×** mais chance, 1ª vs. 3ª classe
**9,39×**, 2ª vs. 3ª classe **3,50×** — todos idênticos aos valores oficiais, como esperado com
`random_state=42` fixo.


**Simulação: a razão de chances é a mesma para todo mundo.** Tome três passageiros diferentes (o homem da 3ª classe, um homem de 30 anos da 1ª classe e o menino da simulação do passo 5). Para cada um, calcule com `predict_proba` a probabilidade como mulher e como homem, mantendo todo o resto. Compare a razão entre as probabilidades e a razão entre as chances.

In [11]:
three = pd.DataFrame({
    'Pclass': [3, 1, 2],
    'Sex': ['male', 'male', 'male'],
    'Age': [25, 30, 8],
    'SibSp': [0, 0, 1],
    'Parch': [0, 0, 2],
    'Fare': [8.05, 80.0, 30.0],
    'Embarked': ['S', 'C', 'S'],
}, index=['homem 3a classe', 'homem 30a 1a classe', 'menino 8a 2a classe'])
three_f = three.copy()
three_f['Sex'] = 'female'

p_m = model.predict_proba(three)[:, 1]
p_f = model.predict_proba(three_f)[:, 1]
odds_m = p_m / (1 - p_m)
odds_f = p_f / (1 - p_f)

res = pd.DataFrame({
    'p (homem)': p_m, 'p (mulher)': p_f,
    'razão de probabilidades (f/m)': p_f / p_m,
    'razão de chances (f/m)': odds_f / odds_m,
}, index=three.index)
print(res.round(4))
print(f"e^(-coef Sex_male) esperado: {np.exp(-coef['cat__Sex_male']):.4f}")


                     p (homem)  p (mulher)  razão de probabilidades (f/m)  \
homem 3a classe         0.0932      0.6010                         6.4487   
homem 30a 1a classe     0.5856      0.9540                         1.6289   
menino 8a 2a classe     0.3258      0.8763                         2.6894   

                     razão de chances (f/m)  
homem 3a classe                     14.6573  
homem 30a 1a classe                 14.6573  
menino 8a 2a classe                 14.6573  
e^(-coef Sex_male) esperado: 14.6573


**Resposta:** A razão entre as probabilidades varia muito de passageiro para passageiro
(6,45× para o homem da 3ª classe, 1,63× para o homem de 30 anos da 1ª classe, 2,69× para o
menino), porque a sigmoide não é linear e o ponto de partida (quão perto de 0 ou 1 já está a
probabilidade do homem) muda o tamanho do salto. Já a razão de chances é **exatamente 14,6573**
nos três casos — idêntica a $e^{-\text{coef(Sex\_male)}}$ calculado no Passo 7 — porque, em
log-odds, trocar `Sex_male` de 1 para 0 soma sempre a mesma constante ao escore z,
independentemente do valor dos outros atributos; e uma soma constante em log-odds é uma
multiplicação constante em chances. Essa é a propriedade central da regressão logística: os
coeficientes multiplicam chances de forma uniforme, não probabilidades.


## Passo 8 — o único teste

**Preveja.** A acurácia de teste vai ficar acima ou abaixo dos 80,9% de treino? Muito ou pouco?

**Resposta (antes de rodar):** Deve ficar abaixo dos 80,9% de treino, porque o modelo nunca
viu esses passageiros e tende a performar um pouco pior fora da amostra usada para ajustá-lo —
mas a queda deve ser pequena (poucos pontos percentuais), já que o modelo é simples (sem
penalidade, poucas features) e não parece estar sobreajustando fortemente.


In [12]:
from sklearn.metrics import confusion_matrix

print(f"acurácia de teste {model.score(X_test, y_test):.3f} · baseline {dummy.score(X_test, y_test):.3f}")
print(confusion_matrix(y_test, model.predict(X_test), labels=[1, 0]))   # positivo primeiro

acurácia de teste 0.804 · baseline 0.615


[[46 23]
 [12 98]]


**Resposta (depois de rodar):** Confirmado, e a queda foi pequena como previsto — acurácia de
teste **80,4%** contra 80,9% de treino (só 0,5 p.p. abaixo), e bem acima da baseline de teste
(**61,5%**). A matriz de confusão (positivo = sobreviveu primeiro) é `[[46, 23], [12, 98]]`: dos
69 sobreviventes reais no teste, 46 foram identificados corretamente e 23 perdidos; dos 110 que
não sobreviveram, 98 foram identificados e 12 teve falso alarme (previsto como sobrevivente).
Esses números batem exatamente com a página (0,804 vs. 0,615; matriz [[46,23],[12,98]]).


**Simulação: mexendo no limiar.** O `predict` usa o limiar 0,5. Nas linhas de *treino* (o teste já foi usado), classifique com os limiares 0,3, 0,4, 0,5, 0,6 e 0,7 e imprima, para cada um, a fração dos sobreviventes encontrados, a fração dos mortos reconhecidos e a acurácia.

In [13]:
p_train = model.predict_proba(X_train)[:, 1]
resultados = []
for thr in [0.3, 0.4, 0.5, 0.6, 0.7]:
    pred = (p_train >= thr).astype(int)
    tp = ((pred == 1) & (y_train == 1)).sum()
    fn = ((pred == 0) & (y_train == 1)).sum()
    tn = ((pred == 0) & (y_train == 0)).sum()
    fp = ((pred == 1) & (y_train == 0)).sum()
    resultados.append({
        'limiar': thr,
        'fração sobreviventes encontrados': tp / (tp + fn),
        'fração mortos reconhecidos': tn / (tn + fp),
        'acurácia': (tp + tn) / len(y_train),
    })
print(pd.DataFrame(resultados).round(3).to_string(index=False))


 limiar  fração sobreviventes encontrados  fração mortos reconhecidos  acurácia
    0.3                             0.824                       0.745     0.775
    0.4                             0.751                       0.804     0.784
    0.5                             0.714                       0.868     0.809
    0.6                             0.652                       0.916     0.815
    0.7                             0.553                       0.964     0.806


**Resposta:** Subir o limiar de 0,3 para 0,7 troca sensibilidade por especificidade: a fração
de sobreviventes encontrados cai de **82,4%** para **55,3%**, enquanto a fração de mortos
reconhecidos sobe de **74,5%** para **96,4%**. A acurácia não é monótona — sobe de **77,5%**
(limiar 0,3) para um máximo de **81,5%** (limiar 0,6) e cai de novo para **80,6%** (limiar 0,7) —
porque a acurácia pondera os dois tipos de erro pela proporção de cada classe (mais mortos que
sobreviventes no treino), então o ponto que maximiza acurácia não é necessariamente o limiar
"neutro" de 0,5. Esses valores (0,775 a 0,815) batem com a faixa citada na página para esse mesmo
exercício.


**Para fechar a aula.** Em uma frase cada: (1) o que um coeficiente da regressão logística multiplica? (2) Por que o menino da simulação do passo 5 recebeu uma probabilidade tão baixa? (3) Por que o conjunto de teste só pode ser usado uma vez?

**Resposta:**

1. Um coeficiente multiplica a versão já pré-processada do atributo (padronizado, se numérico, ou
   o dummy 0/1, se categórico) dentro do escore linear $z$ — o log-odds da sobrevivência; em termos
   de chance, $e^{\text{coef}}$ é o fator que multiplica a chance quando esse atributo (ou dummy)
   aumenta em 1 unidade (ou passa de 0 para 1).
2. O menino da simulação do Passo 5 recebeu probabilidade baixa (entre 38,0% a 2 anos e 5,9% a 60
   anos, sempre abaixo da menina equivalente, que vai de 90,0% a 47,7%) porque o coeficiente de
   `Sex_male` (-2,685) é, em módulo, maior que qualquer efeito plausível da idade sozinha — mesmo
   no extremo mais favorável (bebê de 2 anos), o efeito "ser homem" já consome boa parte do escore
   positivo que a pouca idade geraria.
3. O conjunto de teste só pode ser usado uma vez porque cada vez que ele influencia uma decisão
   (ajustar hiperparâmetros, escolher features, comparar modelos) ele deixa de medir desempenho em
   dados nunca vistos e passa a fazer parte, indiretamente, do processo de ajuste — a acurácia de
   teste reportada deixaria de ser uma estimativa honesta e passaria a estar otimisticamente
   inflada (a mesma lógica de vazamento do Passo 2).


## Sua vez

Para casa, com mais tempo:

1. **A interação.** O passo 1 mostrou que a terceira classe prejudica as mulheres muito mais que os homens. Acrescente a coluna `df['female_3rd'] = ((df['Sex'] == 'female') & (df['Pclass'] == 3)).astype(int)`, coloque-a em `features` e em `categorical` (com `drop='first'`, seu coeficiente compara 1 com 0) e refaça os passos 2 a 7. Leia a razão de chances dela em palavras. Como muda o coeficiente `Sex_male`? Por quê? Para decidir se a coluna ajuda, divida `X_train` de novo em uma parte de treino e uma de validação e compare os dois modelos ali, não no teste.
2. **O limiar.** Nessa validação, calcule `p = model.predict_proba(X_val)[:, 1]` e conte os sobreviventes perdidos e os alarmes falsos para os limiares 0,3, 0,5 e 0,7. Que limiar você usaria se perder um sobrevivente fosse três vezes pior que um alarme falso?
3. **Por que uma categoria de referência.** Monte o pipeline de novo com `OneHotEncoder()` (sem `drop`) e ajuste-o. Depois some 5 aos três pesos de `Pclass` em `model[-1].coef_` e subtraia 5 de `model[-1].intercept_`: as probabilidades previstas em `X_test` mudam? O que isso diz sobre ler um único coeficiente quando toda categoria mantém sua coluna?
4. **Do zero.** Tome `Xt = model[0].transform(X_train)` e rode o laço de gradiente descendente desta aula (pesos e intercepto, \(\eta = 0{,}5\)). Compare com `model[-1].coef_` e `model[-1].intercept_`. Depois de quantas iterações eles concordam em duas casas decimais?

In [14]:
# Exercício 1 — a interação female_3rd
df2 = df.copy()
df2['female_3rd'] = ((df2['Sex'] == 'female') & (df2['Pclass'] == 3)).astype(int)

features2 = features + ['female_3rd']
categorical2 = categorical + ['female_3rd']
X2, y2 = df2[features2], df2['Survived']
X2_train, X2_test, y2_train, y2_test = train_test_split(
    X2, y2, test_size=0.2, stratify=y2, random_state=42)
# nova divisão treino/validação para comparar os dois modelos sem tocar no teste
X2_tr, X2_val, y2_tr, y2_val = train_test_split(
    X2_train, y2_train, test_size=0.2, stratify=y2_train, random_state=42)

preprocess2 = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'),
                          OneHotEncoder(drop='first')), categorical2)])
model_interacao = make_pipeline(preprocess2, LogisticRegression(C=np.inf, max_iter=1000))
model_interacao.fit(X2_tr, y2_tr)

model_sem_interacao = make_pipeline(
    ColumnTransformer([
        ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric),
        ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'),
                              OneHotEncoder(drop='first')), categorical)]),
    LogisticRegression(C=np.inf, max_iter=1000))
model_sem_interacao.fit(X2_tr[features], y2_tr)

names_i = model_interacao[0].get_feature_names_out()
coef_i = pd.Series(model_interacao[-1].coef_[0], index=names_i)
print(pd.DataFrame({'coef': coef_i, 'e^coef': np.exp(coef_i)}).round(3))
print()
print(f"acurácia validação, sem interação: {model_sem_interacao.score(X2_val[features], y2_val):.3f}")
print(f"acurácia validação, com interação: {model_interacao.score(X2_val, y2_val):.3f}")
print(f"coef Sex_male sem interação (mesmo treino): {model_sem_interacao[-1].coef_[0][list(model_sem_interacao[0].get_feature_names_out()).index('cat__Sex_male')]:.3f}")
print(f"coef Sex_male com interação: {coef_i['cat__Sex_male']:.3f}")


                    coef  e^coef
num__Age          -0.610   0.544
num__SibSp        -0.233   0.792
num__Parch        -0.015   0.985
num__Fare          0.094   1.099
cat__Pclass_2     -1.757   0.173
cat__Pclass_3     -1.944   0.143
cat__Sex_male     -4.534   0.011
cat__Embarked_Q    0.694   2.001
cat__Embarked_S   -0.390   0.677
cat__female_3rd_1 -2.709   0.067

acurácia validação, sem interação: 0.769
acurácia validação, com interação: 0.783
coef Sex_male sem interação (mesmo treino): -2.770
coef Sex_male com interação: -4.534


**Resposta:** A nova coluna `female_3rd` tem coeficiente **-2,709** (e^coef = **0,067**): ser
mulher *e* estar na 3ª classe, ao mesmo tempo, multiplica a chance por 0,067 **além** dos efeitos
já contados separadamente por `Sex_male` e `Pclass_3` — ou seja, a penalidade de estar na 3ª
classe é bem maior para mulheres do que a soma dos dois efeitos isolados sugeriria, confirmando o
padrão visto no Passo 1 (mulheres da 3ª classe, 50,0%, muito abaixo das demais mulheres). O
coeficiente de `Sex_male` fica muito mais negativo com a interação (**-4,534** vs. **-2,770** no
modelo sem interação, treinado no mesmo recorte de treino): sem a coluna de interação, o
coeficiente de sexo tinha que "absorver" parte da penalidade extra das mulheres da 3ª classe
médio entre as classes; com a interação isolando esse efeito, o coeficiente de sexo passa a
refletir só o efeito de ser homem nas classes 1ª/2ª, que é mais extremo. Na validação, o modelo
com interação tem acurácia **78,3%** contra **76,9%** do modelo sem interação (+1,4 p.p.) — uma
pequena melhora que sugere que vale manter a interação, embora a diferença seja pequena o
suficiente para merecer confirmação com mais dados ou validação cruzada.


In [15]:
# Exercício 2 — limiar na validação
p_val = model_interacao.predict_proba(X2_val)[:, 1]
resultados2 = []
for thr in [0.3, 0.5, 0.7]:
    pred = (p_val >= thr).astype(int)
    missed_survivors = int(((pred == 0) & (y2_val == 1)).sum())
    false_alarms = int(((pred == 1) & (y2_val == 0)).sum())
    custo = 3 * missed_survivors + false_alarms
    resultados2.append({'limiar': thr, 'sobreviventes perdidos': missed_survivors,
                         'falsos alarmes': false_alarms,
                         'custo (perder=3x alarme)': custo})
print(f"n validação: {len(y2_val)}")
print(pd.DataFrame(resultados2).to_string(index=False))


n validação: 143
 limiar  sobreviventes perdidos  falsos alarmes  custo (perder=3x alarme)
    0.3                      14              17                        59
    0.5                      21              10                        73
    0.7                      26               4                        82


**Resposta:** Na validação (143 passageiros), o limiar 0,3 perde **14** sobreviventes e gera
**17** falsos alarmes; o limiar 0,5 perde **21** e gera **10**; o limiar 0,7 perde **26** e gera
só **4**. Se perder um sobrevivente custa 3× um falso alarme, o custo total (3×perdidos +
alarmes) é 3×14+17=**59** no limiar 0,3, 3×21+10=**73** no limiar 0,5 e 3×26+4=**82** no limiar
0,7 — o limiar **0,3** é o mais barato dos três nesse critério, porque ele é o que perde menos
sobreviventes (o erro caro), mesmo trocando por mais falsos alarmes (o erro barato). Isso mostra
que o limiar certo depende do custo relativo dos dois erros, não só da acurácia.


In [16]:
# Exercício 3 — por que uma categoria de referência
preprocess3 = ColumnTransformer([
    ('num', make_pipeline(SimpleImputer(strategy='median'), StandardScaler()), numeric),
    ('cat', make_pipeline(SimpleImputer(strategy='most_frequent'), OneHotEncoder()), categorical)])
model_sem_drop = make_pipeline(preprocess3, LogisticRegression(C=np.inf, max_iter=1000))
model_sem_drop.fit(X_train, y_train)

names3 = list(model_sem_drop[0].get_feature_names_out())
pclass_idx = [i for i, n in enumerate(names3) if n.startswith('cat__Pclass')]
print("colunas de Pclass:", [names3[i] for i in pclass_idx])

proba_antes = model_sem_drop.predict_proba(X_test)[:, 1].copy()
model_sem_drop[-1].coef_[0, pclass_idx] += 5
model_sem_drop[-1].intercept_[0] -= 5
proba_depois = model_sem_drop.predict_proba(X_test)[:, 1]

print(f"maior |mudança| de probabilidade em X_test: {np.max(np.abs(proba_depois - proba_antes)):.2e}")
print("5 primeiras probabilidades antes:", proba_antes[:5].round(4))
print("5 primeiras probabilidades depois:", proba_depois[:5].round(4))


colunas de Pclass: ['cat__Pclass_1', 'cat__Pclass_2', 'cat__Pclass_3']
maior |mudança| de probabilidade em X_test: 3.89e-16
5 primeiras probabilidades antes: [0.0627 0.0434 0.1489 0.0323 0.6837]
5 primeiras probabilidades depois: [0.0627 0.0434 0.1489 0.0323 0.6837]


**Resposta:** Mesmo somando 5 aos três pesos de `Pclass` e subtraindo 5 do intercepto, a maior
mudança em qualquer probabilidade prevista em `X_test` foi de **≈4×10⁻¹⁶** — ou seja, nenhuma
mudança real (é só erro de ponto flutuante). Isso acontece porque, sem categoria descartada,
todo passageiro tem exatamente um dummy de `Pclass` igual a 1 e dois iguais a 0; somar a mesma
constante aos três pesos sempre soma exatamente essa constante ao escore z de qualquer
passageiro (pois um, e só um, dos três dummies está ativo), e subtrair a mesma constante do
intercepto cancela exatamente esse deslocamento. Isso mostra por que ler **um único** coeficiente
de uma variável categórica sem categoria de referência não tem significado isolado — qualquer
coeficiente de uma categoria pode ser deslocado por uma constante arbitrária (compensada no
intercepto) sem mudar nenhuma previsão; só **diferenças** entre coeficientes de categorias da
mesma variável (ou o uso de uma referência fixa, como faz `drop='first'`) são interpretáveis.


In [17]:
# Exercício 4 — do zero: gradiente descendente
Xt = model[0].transform(X_train)
if hasattr(Xt, "toarray"):
    Xt = Xt.toarray()
yt = y_train.values
n, d = Xt.shape

w_gd = np.zeros(d)
b_gd = 0.0
eta = 0.5
w_sklearn, b_sklearn = model[-1].coef_[0], model[-1].intercept_[0]

match_iter = None
max_iters = 20000
for it in range(1, max_iters + 1):
    z = Xt @ w_gd + b_gd
    p = 1 / (1 + np.exp(-z))
    grad_w = Xt.T @ (p - yt) / n
    grad_b = np.mean(p - yt)
    w_gd -= eta * grad_w
    b_gd -= eta * grad_b
    if match_iter is None and np.allclose(np.round(w_gd, 2), np.round(w_sklearn, 2)) \
            and round(b_gd, 2) == round(b_sklearn, 2):
        match_iter = it
        break

print("coeficientes sklearn:", np.round(w_sklearn, 3))
print("coeficientes GD     :", np.round(w_gd, 3))
print(f"intercepto sklearn: {b_sklearn:.3f} · GD: {b_gd:.3f}")
print(f"iterações até concordarem em 2 casas decimais: {match_iter}")


coeficientes sklearn: [-0.513 -0.267 -0.072  0.108 -0.987 -2.239 -2.685  0.32  -0.417]
coeficientes GD     : [-0.513 -0.267 -0.072  0.108 -0.985 -2.238 -2.684  0.32  -0.418]
intercepto sklearn: 2.782 · GD: 2.781
iterações até concordarem em 2 casas decimais: 2699


**Resposta:** O laço de gradiente descendente (η=0,5, partindo de pesos e intercepto zerados)
converge para coeficientes praticamente idênticos aos do `LogisticRegression(C=np.inf)` do
scikit-learn — por exemplo, `Sex_male` -2,684 (GD) vs. -2,685 (sklearn), `Pclass_3` -2,238 vs.
-2,239, intercepto 2,781 vs. 2,782 — porque, sem penalidade (C=np.inf), a regressão logística é
exatamente o problema de otimização convexa que o gradiente descendente está resolvendo; os dois
caminhos diferentes (fórmula fechada via otimizador do sklearn vs. passos iterativos) convergem
para o mesmo mínimo da entropia cruzada. Foram necessárias **2.699** iterações para que todos os
9 coeficientes e o intercepto batessem com o resultado do sklearn arredondados a duas casas
decimais — um número grande porque η=0,5 é um passo conservador e a convergência de gradiente
descendente simples (sem momentum/Adam) perto do mínimo costuma ser lenta.


## Conclusões

- O pipeline (regressão logística sem penalidade, `C=np.inf`, sobre 9 colunas pré-processadas)
  chega a **80,9%** de acurácia de treino e **80,4%** de teste, bem acima da baseline
  (`DummyClassifier`, 61,7%/61,5%) — e esses quatro números batem exatamente com a página da
  aula, confirmando que o split e o ajuste são reprodutíveis com `random_state=42` fixo.
- Sexo domina a previsão: a razão de chances mulher/homem é **14,66×**, maior em módulo que 1ª vs.
  3ª classe (**9,39×**) e muito maior que o efeito de um desvio-padrão de idade (**0,599×**,
  13 anos) — na simulação da idade, nenhum menino supera a probabilidade da menina equivalente em
  nenhuma idade testada (2 a 60 anos).
- A razão de chances $e^{\text{coef}}$ é uma propriedade **constante** do modelo — na simulação
  do Passo 7, trocar o sexo multiplicou a chance de sobreviver por exatamente **14,6573** em três
  passageiros muito diferentes — enquanto a razão de *probabilidades* variou de 1,63× a 6,45× entre
  eles, por causa da não linearidade da sigmoide.
- A matriz de confusão de teste, `[[46, 23], [12, 98]]`, mostra que o modelo erra mais perdendo
  sobreviventes (23 falsos negativos) do que soando falso alarme (12 falsos positivos); variar o
  limiar de decisão de 0,3 a 0,7 move esse equilíbrio (de 82,4% a 55,3% de sobreviventes
  encontrados) sem necessariamente melhorar a acurácia (que varia de 77,5% a 81,5%, com máximo em
  0,6, não em 0,5).
- Acrescentar a interação `female_3rd` isola um efeito que o modelo simples sem ela tinha que
  distribuir entre os coeficientes de sexo e classe (coeficiente de `Sex_male` salta de -2,770
  para -4,534 quando a interação entra), e melhora a acurácia de validação de 76,9% para 78,3%
  — uma evidência (pequena, mas na direção certa) de que a penalidade da 3ª classe realmente é
  mais severa para mulheres do que para homens.
- O gradiente descendente implementado do zero (η=0,5) converge para os mesmos coeficientes do
  `scikit-learn` em 2.699 iterações (duas casas decimais) — prova de que, sem regularização, a
  regressão logística do sklearn está resolvendo exatamente o mesmo problema de otimização convexa
  que o laço manual desta aula.
